# Phase 3: LSTM and comparison

Feed sequences of the last 7–30 days of price into a small LSTM to predict next-day price. Same train/val/test split and metrics as 01 and 02 for a fair comparison. Results table: Baselines vs Lag vs LSTM.

In [ ]:
# Colab: run the next cell (it clones + installs + imports).

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import subprocess
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

if 'google.colab' in sys.modules:
    repo_dir = Path('/content/crypto-price-prediction')
    if not (repo_dir / 'src').exists():
        if repo_dir.exists():
            import shutil
            shutil.rmtree(repo_dir)
        subprocess.run(['git', 'clone', '-q', 'https://github.com/MOONx02/crypto-price-prediction.git', str(repo_dir)], check=True)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(repo_dir / 'requirements.txt')], check=True)
    ROOT = repo_dir
else:
    ROOT = Path('.').resolve()
    if ROOT.name == 'notebooks':
        ROOT = ROOT.parent
    elif ROOT.name != 'crypto-price-prediction':
        ROOT = ROOT / 'crypto-price-prediction'
sys.path.insert(0, str(ROOT))
from src.metrics import regression_metrics

## 1. Load data and time-based split (same as 01 and 02)

In [2]:
DATA_DIR = ROOT / 'data'
DATA_DIR.mkdir(exist_ok=True)
cache_path = DATA_DIR / 'BTC_USD_daily.parquet'
if not cache_path.exists():
    import yfinance as yf
    raw = yf.download('BTC-USD', start='2017-01-01', end=None, progress=False, auto_adjust=True)
    if raw.index.nlevels > 1:
        raw = raw.reset_index(level=1, drop=True)
    raw.index = pd.to_datetime(raw.index).tz_localize(None)
    raw = raw.sort_index().ffill().dropna()
    df = raw[['Close']].copy()
    df.columns = ['price']
    if 'Volume' in raw.columns:
        df['volume'] = raw['Volume']
    df.to_parquet(cache_path)
    print('Downloaded and saved', cache_path)
df = pd.read_parquet(cache_path)
n = len(df)
train_end = int(0.70 * n)
val_end = int(0.85 * n)
train_df = df.iloc[:train_end]
val_df = df.iloc[train_end:val_end]
test_df = df.iloc[val_end:]
price = df['price'].values.astype(np.float32)
T = len(price)
print(df.shape, '| Train', len(train_df), 'Val', len(val_df), 'Test', len(test_df))

Train 0 -> 2338   Val 2338 -> 2839   Test 2839 -> 3340


## 2. Build sequences for LSTM (3.1)

**Input:** last `SEQ_LEN` days of price (one feature). **Target:** next-day price.

Shape: `(samples, timesteps, features)` = `(N, SEQ_LEN, 1)` for Keras LSTM. No future leakage: each sample uses only past data.

In [3]:
SEQ_LEN = 30

def build_seq(price, start, end, seq_len):
    X = np.array([price[i - seq_len : i] for i in range(start, end)], dtype=np.float32).reshape(-1, seq_len, 1)
    y = price[start:end]
    return X, y

X_tr, y_tr = build_seq(price, SEQ_LEN, train_end, SEQ_LEN)
X_va, y_va = build_seq(price, train_end, val_end, SEQ_LEN)
X_te, y_te = build_seq(price, val_end, T, SEQ_LEN)
print('X_tr', X_tr.shape, 'y_tr', y_tr.shape)
print('X_va', X_va.shape, 'X_te', X_te.shape)

X_train (2308, 30, 1) y_train (2308,)
X_val   (501, 30, 1) y_val (501,)
X_test  (501, 30, 1) y_test (501,)


## 3. LSTM model (3.2)

Small stack: 1–2 LSTM layers, then Dense(1) for regression. Same task: next-day price.

In [4]:
model = keras.Sequential([
    layers.Input(shape=(SEQ_LEN, 1)),
    layers.LSTM(32, return_sequences=True),
    layers.LSTM(16),
    layers.Dense(1)
])
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 30, 32)         │         4,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 16)             │         3,136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,505 (29.32 KB)

 Trainable params: 7,505 (29.32 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
early = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=10, restore_best_weights=True
)
history = model.fit(
    X_tr, y_tr,
    validation_data=(X_va, y_va),
    epochs=80,
    batch_size=32,
    callbacks=[early],
    verbose=1
)

Epoch 1/80


In [ ]:
plt.figure(figsize=(8, 3))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.legend()
plt.title('Loss (MSE)')
plt.subplot(1, 2, 2)
plt.plot(history.history['mae'], label='train')
plt.plot(history.history['val_mae'], label='val')
plt.legend()
plt.title('MAE')
plt.tight_layout()
plt.show()

## 4. Evaluate on test set (3.3)

Same test period and metrics (MAE, RMSE, directional accuracy) as baselines and lag model.

In [ ]:
pred_lstm = model.predict(X_te, verbose=0).ravel()
m_lstm = regression_metrics(y_te, pred_lstm)
print('LSTM:', m_lstm)

## 5. Results table and summary (3.4)

Fill baseline and lag numbers from 01 and 02 (same test period). Short conclusion: which model wins on which metric.

In [ ]:
# Replace with your numbers from 01_data_and_baselines and 02_lag_model (same test set)
results = pd.DataFrame({
    'Model': ['Last value (baseline)', '7-day MA (baseline)', 'Lag + Ridge (02)', 'LSTM (this notebook)'],
    'MAE': [np.nan, np.nan, np.nan, m_lstm['mae']],
    'RMSE': [np.nan, np.nan, np.nan, m_lstm['rmse']],
    'Dir.Acc': [np.nan, np.nan, np.nan, m_lstm['directional_accuracy']]
})
results

In [ ]:
# After filling MAE/RMSE/Dir.Acc from 01 and 02, uncomment and run:
# results = results.fillna(...)  # or set values in the DataFrame above
# print(results.to_string(index=False))
# Summary: e.g. "LSTM improves MAE over baselines but may underperform the lag model; directional accuracy ..."

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(pred_lstm, y_te, alpha=0.5, s=15)
ax.plot([y_te.min(), y_te.max()], [y_te.min(), y_te.max()], 'r--', label='y=x')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('LSTM: Predicted vs actual (test)')
ax.legend()
plt.tight_layout()
plt.show()